In [14]:
# --- CI-safe bootstrap & paths ---
import os
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# детерминизм
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# безопасные пути для CI/локально
CWD = Path.cwd()
REPORTS = (CWD.parent / "reports") if CWD.name == "notebooks" else (CWD / "reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print({"cwd": str(CWD), "reports": str(REPORTS)})


{'cwd': 'c:\\Users\\Natalia\\PycharmProjects\\trailflow\\notebooks', 'reports': 'c:\\Users\\Natalia\\PycharmProjects\\trailflow\\reports'}


In [15]:
# --- Bootstrap: make the notebook CI-safe and reproducible ---# --- Synthetic data & ROI curve ---
base_risk = 0.12
x1 = np.random.normal(0, 1, 3000)
x2 = np.random.normal(0, 1, 3000)
logit = np.log(base_risk/(1-base_risk)) + 0.5 * x1 + 0.2 * x2 + np.random.normal(0, 0.5, 3000)
p = 1 / (1 + np.exp(-logit))
y = np.random.binomial(1, p)

Ks = list(range(5, 55, 5))

def enb_from_scores(y_true, p_pred, k_percent, cost_no_show=200.0, cost_intervention=1.5, uplift=0.30):
    n = len(y_true)
    m = max(1, int(n * (k_percent / 100.0)))
    order = np.argsort(-p_pred)
    p_top = np.asarray(p_pred)[order][:m]
    benefit = uplift * float(p_top.sum()) * cost_no_show
    cost = m * cost_intervention
    return benefit - cost

enb = [enb_from_scores(y, p, k) for k in Ks]

plt.figure()
plt.plot(Ks, enb, marker="o")
plt.xlabel("Top-K% target budget")
plt.ylabel("Expected Net Benefit (€)")
plt.title("ROI curve (synthetic)")
out_path = REPORTS / "roi_curve_from_notebook.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
print("Saved figure to:", out_path)


Saved figure to: c:\Users\Natalia\PycharmProjects\trailflow\reports\roi_curve_from_notebook.png


In [16]:
# --- Minimal synthetic data for ROI curve (no external files) ---
# Simulate probabilities with mild signal so that ENB curve is meaningful
base_risk = 0.12
x1 = np.random.normal(0, 1, N)
x2 = np.random.normal(0, 1, N)
logit = (
    np.log(base_risk / (1 - base_risk))
    + 0.5 * x1
    + 0.2 * x2
    + np.random.normal(0, 0.5, N)
)
p = 1 / (1 + np.exp(-logit))
y = np.random.binomial(1, p)

# Build ROI curve for K in 5..50%
Ks = list(range(5, 55, 5))
enb = [enb_from_scores(y, p, k) for k in Ks]

plt.figure()
plt.plot(Ks, enb, marker="o")
plt.xlabel("Top-K% target budget")
plt.ylabel("Expected Net Benefit (€)")
plt.title("ROI curve (synthetic)")
plt.tight_layout()


out_path = REPORTS / "roi_curve_from_notebook.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
print("Saved figure to:", out_path)
# plt.savefig("reports/roi_curve_from_notebook.png", dpi=120)
enb[:3], len(enb)

Saved figure to: c:\Users\Natalia\PycharmProjects\trailflow\reports\roi_curve_from_notebook.png


([21742.171393955206, 37308.64739859423, 50282.97175657193], 10)